# Threads Scraper Analysis Stack 

---

## 架構總覽

```
ScrapeCreators API (/v1/threads/*)
        │  HTTP GET + x-api-key
        ▼
┌─ threads_client.py ──────────────────────┐
│  ThreadsClient._request()   (重試 + 錯誤處理)  │
│  ├─ search_posts()    → list[ThreadsPost]     │
│  ├─ get_profile()     → ThreadsProfile        │
│  ├─ get_user_posts()  → list[ThreadsPost]     │
│  ├─ get_post()        → ThreadsPost           │
│  └─ search_users()    → list[dict]            │
│                                               │
│  ThreadsPost.from_api_response()              │
│  └─ 原始 JSON → 正規化 dataclass               │
└───────────────┬───────────────────────┘
                ▼
┌─ keyword_monitor.py ─────────────────────┐
│  KeywordMonitor                               │
│  ├─ search_keyword()  (去重 by post_id)       │
│  ├─ run_round()       (批次搜多關鍵字)         │
│  └─ export_results()  (CSV + JSON → data/)    │
└───────────────┬───────────────────────┘
                ▼
┌─ Notebook (本檔案) ──────────────────────┐
│  載入 → 語言過濾 → 受眾分群 → 品牌分析        │
└──────────────────────────────────────────┘
```

## Pipeline 節點

| # | 節點 | 說明 |
|---|------|------|
| 1 | 環境設置 | 匯入套件、設定 |
| 2 | API 串接 | 初始化 ThreadsClient、驗證連線 |
| 3 | 爬蟲執行 | 用 KeywordMonitor 批次爬取關鍵字貼文 |
| 4 | 資料 Extract | 檢視 API 原始回傳 → dataclass 解析 → 匯出 |
| 5 | 載入 & 過濾 | 讀取 CSV、繁體中文過濾、清洗 |
| 6 | 受眾輪廓 | 高相關貼文、活躍帳號、關鍵字互動 |
| 7 | 受眾分群 | 求推薦 vs 分享心得、品牌/裝備品類分析 |
| 8 | 洞察總結 | 受眾報告 |

## 節點 1：環境設置

匯入所有需要的套件，包括本專案的 `threads_client` 和 `keyword_monitor`。

In [43]:
import os
import sys
from pathlib import Path

# 從 shared/ 導入共用模組（相對路徑以 notebook 位置為基準）
SHARED_DIR = (Path.cwd() / ".." / "shared").resolve()
sys.path.insert(0, str(SHARED_DIR))

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
print(f"  - shared 路徑: {SHARED_DIR}")
print(f"  - 輸出目錄: {DATA_DIR.resolve()}")


  - shared 路徑: I:\Fnte Workdir\threads-audience-discovery-stack\shared
  - 輸出目錄: I:\Fnte Workdir\threads-audience-discovery-stack\vn_visa_service\data


In [44]:

import json
import re
import time

from collections import Counter
from datetime import datetime

import pandas as pd
import numpy as np

# 匯入自定模組
from threads_client import (
    ThreadsClient, ThreadsPost, ThreadsProfile,
    posts_to_dicts, save_json, save_csv,
)
from keyword_monitor import KeywordMonitor


In [45]:
from dotenv import load_dotenv
load_dotenv(Path(".env"), override=True)

True

## 節點 2：API 串接

初始化 `ThreadsClient`，驗證 API Key 與連線狀態。

**API 架構：**
- 底層使用 `requests.Session` 保持連線
- 所有請求透過 `_request()` 統一處理：自動重試（429/500）、錯誤分類（401/402/400）
- Header 帶 `x-api-key` 認證

In [46]:
# ── 初始化 API Client ──
# API Key 設定
API_KEY = os.environ.get("SCRAPECREATORS_API_KEY")

client = ThreadsClient(api_key=API_KEY, timeout=30, max_retries=3)
print("✔ ThreadsClient 初始化成功")
print(f"  - Base URL: https://api.scrapecreators.com")
print(f"  - Timeout: 30s, Max retries: 3")

# 驗證連線：查詢剩餘額度
try:
    balance = client.get_credit_balance()
    print(f"  - API 額度剩餘: {balance} credits")
except Exception as e:
    print(f"  - ⚠ 額度查詢失敗: {e}")

✔ ThreadsClient 初始化成功
  - Base URL: https://api.scrapecreators.com
  - Timeout: 30s, Max retries: 3
  - API 額度剩餘: 23751 credits


In [47]:
# # ── 逐一測試 5 個 API Endpoint ──
# # 每次呼叫消耗 1 credit

# print("=" * 55)
# print("  測試 5 個 Threads API Endpoints")
# print("=" * 55)

# # ── Endpoint 1: 搜尋貼文（核心爬蟲 endpoint）──
# print("\n── 1. search_posts('AI') ──")
# print("   GET /v1/threads/search?query=AI")
# test_posts = client.search_posts(query="AI")
# print(f"   回傳 {len(test_posts)} 筆貼文")
# if test_posts:
#     p = test_posts[0]
#     print(f"   範例: @{p.username} | {p.like_count} likes")
#     print(f"   Text: {p.text[:80]}...")

# # ── Endpoint 2: 用戶 Profile ──
# print("\n── 2. get_profile('zuck') ──")
# print("   GET /v1/threads/profile?handle=zuck")
# try:
#     profile = client.get_profile("zuck")
#     print(f"   @{profile.username} | {profile.full_name}")
#     print(f"   Followers: {profile.follower_count:,} | Verified: {profile.is_verified}")
# except Exception as e:
#     print(f"   Error: {e}")

# # ── Endpoint 3: 用戶貼文列表 ──
# print("\n── 3. get_user_posts('zuck') ──")
# print("   GET /v1/threads/user/posts?handle=zuck")
# try:
#     user_posts = client.get_user_posts("zuck")
#     print(f"   回傳 {len(user_posts)} 筆貼文")
# except Exception as e:
#     print(f"   Error: {e}")

# # ── Endpoint 4: 搜尋用戶 ──
# print("\n── 4. search_users('tech') ──")
# print("   GET /v1/threads/search/users?query=tech")
# try:
#     users = client.search_users("tech")
#     print(f"   回傳 {len(users)} 位用戶")
#     for u in users[:3]:
#         print(f"   @{u.get('username', '?')} — {u.get('follower_count', '?')} followers")
# except Exception as e:
#     print(f"   Error: {e}")

# # ── Endpoint 5: 單篇貼文詳情 ──
# print("\n── 5. get_post(url) ──")
# print("   GET /v1/threads/post?url=...")
# if test_posts and test_posts[0].permalink:
#     try:
#         single = client.get_post(test_posts[0].permalink)
#         print(f"   @{single.username} | {single.like_count} likes | {single.reply_count} replies")
#     except Exception as e:
#         print(f"   Error: {e}")

# print("\n" + "=" * 55)

## 節點 3：爬蟲主程式


使用 `KeywordMonitor` 批次爬取多個關鍵字的貼文。

**執行流程：**
```
KeywordMonitor.run_round(keywords)
  → 逐一呼叫 client.search_posts(keyword)
    → _request() → GET /v1/threads/search?query=...
    → API JSON → ThreadsPost.from_api_response() 逐筆解析
  → seen_ids 去重（跨關鍵字 + 跨輪次）
  → 累積到 all_posts[keyword]
  → export_results() → CSV + JSON
```

In [48]:
# 5 個關鍵字 = 5 credits / 輪
KEYWORDS = [
    "越南簽證",
    "越南電子簽",
    "越南evisa",
    "越南快速通關",
    "快速通關",
]
PROJECT_TAG = "vietnam_visa"

In [49]:
# ── 爬蟲模式設定 ──
# MODE = "range"    → 自訂日期區間（手動回測、一次性分析）
# MODE = "schedule" → 模擬 Airflow 排程時段（測試 DAG 邏輯）
# MODE = "day"      → 單日爬取（快速測試）
MODE = "day"

# --- range 模式參數 ---
RANGE_START = "2026-04-02"
RANGE_END   = "2026-04-06"

# --- day 模式參數 ---
DAY_DATE = "2026-04-19"

# --- schedule 模式參數（模擬排程）---
SIMULATE_HOUR = 10  # 模擬時段: 10, 14, 18

# --- 動態輪數設定 ---
USE_ADAPTIVE = False   # True = run_adaptive (自動多輪), False = 固定輪數
FIXED_ROUNDS = 1       # 僅 USE_ADAPTIVE=False 時生效
MIN_ROUNDS = 2         # 僅 USE_ADAPTIVE=True 時生效
MAX_ROUNDS = 4
DUP_THRESHOLD = 0.80   # 當重複率超過此值，且新增數趨緩時，停止後續輪數（僅 adaptive 模式）

# ── get_scrape_params ──
from datetime import timedelta
from zoneinfo import ZoneInfo

TZ_TPE = ZoneInfo("Asia/Taipei")
SCHEDULE_HOURS = [10, 14, 18]

def get_scrape_params(execution_hour: int, ref_date=None):
    """
    根據排程時段計算 API 日期範圍 + post-filter 時間窗口。
    與 fetch_threads_posts.py 中的邏輯一致。
    """
    from datetime import date
    if ref_date is None:
        ref_date = date.today()

    today_str = ref_date.strftime('%Y-%m-%d')
    yesterday_str = (ref_date - timedelta(days=1)).strftime('%Y-%m-%d')
    ref_dt = datetime.combine(ref_date, datetime.min.time()).replace(tzinfo=TZ_TPE)

    closest_hour = min(SCHEDULE_HOURS, key=lambda h: abs(h - execution_hour))

    if closest_hour == 10:
        return {
            "start_date": yesterday_str,
            "end_date": today_str,
            "time_window": (ref_dt.replace(hour=0), ref_dt.replace(hour=10)),
            "label": f"10:00 slot | API {yesterday_str}~{today_str}, filter 00:00~10:00",
        }
    else:
        idx = SCHEDULE_HOURS.index(closest_hour)
        prev_hour = SCHEDULE_HOURS[idx - 1]
        return {
            "start_date": today_str,
            "end_date": today_str,
            "time_window": (ref_dt.replace(hour=prev_hour), ref_dt.replace(hour=closest_hour)),
            "label": f"{closest_hour}:00 slot | API {today_str}, filter {prev_hour}:00~{closest_hour}:00",
        }

# ── 根據 MODE 決定參數 ──
if MODE == "schedule":
    params = get_scrape_params(SIMULATE_HOUR)
    START_DATE = params["start_date"]
    END_DATE = params["end_date"]
    TIME_WINDOW = params["time_window"]
    print(f"[schedule] {params['label']}")
    print(f"  time_window: {TIME_WINDOW[0].isoformat()} ~ {TIME_WINDOW[1].isoformat()}")
elif MODE == "day":
    START_DATE = DAY_DATE
    END_DATE = DAY_DATE
    TIME_WINDOW = None
    print(f"[day] {DAY_DATE}")
else:  # range
    START_DATE = RANGE_START
    END_DATE = RANGE_END
    TIME_WINDOW = None
    print(f"[range] {RANGE_START} ~ {RANGE_END}")

print(f"  API params: start_date={START_DATE}, end_date={END_DATE}")
print(f"  adaptive: {'ON' if USE_ADAPTIVE else 'OFF'} (rounds={FIXED_ROUNDS if not USE_ADAPTIVE else f'{MIN_ROUNDS}~{MAX_ROUNDS}'})")
# ── 執行關鍵字爬蟲 ──
monitor = KeywordMonitor(client, output_dir=str(DATA_DIR))

print(f"爬取關鍵字: {len(KEYWORDS)} 組")
print(f"專案標記: {PROJECT_TAG}")
print(f"輸出目錄: {DATA_DIR}")
print(f"預計 API 請求數: {len(KEYWORDS)} 次/輪（每關鍵字 1 次）")
print(f"預計 credits 消耗: 約 {len(KEYWORDS)} credits/輪（以 Credits remaining 實際變化為準）\n")

# ── 執行爬蟲 ──
all_round_stats = []

if USE_ADAPTIVE:
    print(f"[adaptive] min={MIN_ROUNDS}, max={MAX_ROUNDS}, dup_threshold={DUP_THRESHOLD}")
    print("[adaptive] 用途：當新貼文趨緩且重複率升高時，自動停止後續輪數，降低 credits 消耗")
    all_posts, all_round_stats = monitor.run_adaptive(
        KEYWORDS,
        start_date=START_DATE,
        end_date=END_DATE,
        min_rounds=MIN_ROUNDS,
        max_rounds=MAX_ROUNDS,
        dup_threshold=DUP_THRESHOLD,
    )
else:
    for round_num in range(FIXED_ROUNDS):
        print(f"{'='*40} 第 {round_num+1}/{FIXED_ROUNDS} 輪 {'='*40}")
        _results, _stats = monitor.run_round(KEYWORDS, start_date=START_DATE, end_date=END_DATE)
        all_round_stats.append(_stats)
        print(f"  API回傳總筆數={_stats['api_total']}, 新增筆數={_stats['new_count']}, 重複率={_stats['dup_ratio']:.1%}")

# ── 合併所有關鍵字，加上 search_keyword 欄，去重 ──
rows = []
for kw, posts in monitor.all_posts.items():
    for p in posts:
        d = posts_to_dicts([p])[0]
        d['search_keyword'] = kw
        rows.append(d)

df_out = pd.DataFrame(rows).drop_duplicates(subset='post_id', keep='first')

# ── 時間窗口 post-filter（僅 schedule 模式）──
if TIME_WINDOW is not None:
    df_out['timestamp'] = pd.to_datetime(df_out['timestamp'], utc=True)
    start_utc = TIME_WINDOW[0].astimezone(ZoneInfo("UTC"))
    end_utc = TIME_WINDOW[1].astimezone(ZoneInfo("UTC"))
    before = len(df_out)
    df_out = df_out[(df_out['timestamp'] >= start_utc) & (df_out['timestamp'] < end_utc)].copy()
    print(f"\n[time_window filter] {before} -> {len(df_out)} posts")

print(f"\n── 爬取完成 ──")
print(f"  去重後貼文: {len(df_out)} 篇")
print(f"  總輪數: {len(all_round_stats)}")
total_api_results = sum(s.get('api_total', 0) for s in all_round_stats)
estimated_credits = len(KEYWORDS) * len(all_round_stats)
print(f"  API回傳總筆數(各輪加總): {total_api_results}")
print(f"  估算 credits 消耗: 約 {estimated_credits}（以 Credits remaining 差值為準）")

# ── 多輪策略覆蓋率提升評估  ── 
round_new = [s.get('new_count', 0) for s in all_round_stats]
if round_new:
    baseline = round_new[0]
    final_cum = sum(round_new)
    lift_pct = ((final_cum - baseline) / baseline * 100) if baseline > 0 else 0.0

    print(f"  第1輪新增: {baseline}")
    print(f"  多輪累積新增: {final_cum}")
    print(f"  覆蓋率提升(相對第1輪): {lift_pct:.1f}%")
    print("  各輪明細:")
    for i, s in enumerate(all_round_stats, 1):
        print(f"    R{i}: api={s['api_total']}, new={s['new_count']}, dup={s['dup_ratio']:.1%}")
#  ── 

# # ── 原始資料立即儲存 ──
# raw_csv = DATA_DIR / f"threads_{PROJECT_TAG}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
# raw_json = raw_csv.with_suffix('.json')
# df_out.to_csv(raw_csv, index=False, encoding='utf-8')
# save_json(df_out.to_dict(orient='records'), str(raw_json))
# print(f"  {raw_csv.name}")
# print(f"  {raw_json.name}")

2026-04-20 09:37:23 [INFO] ─── Search round: 2026-04-20 09:37:23 ───


[day] 2026-04-19
  API params: start_date=2026-04-19, end_date=2026-04-19
  adaptive: OFF (rounds=1)
爬取關鍵字: 5 組
專案標記: vietnam_visa
輸出目錄: data
預計 API 請求數: 5 次/輪（每關鍵字 1 次）
預計 credits 消耗: 約 5 credits/輪（以 Credits remaining 實際變化為準）

======================================== 第 1/1 輪 ========================================


2026-04-20 09:37:29 [INFO] Credits remaining: 23750
2026-04-20 09:37:29 [INFO]   '越南簽證': 8 results, 7 new (total unique: 7)
2026-04-20 09:37:31 [INFO] Credits remaining: 23749
2026-04-20 09:37:31 [INFO]   '越南電子簽': 0 results, 0 new (total unique: 0)
2026-04-20 09:37:33 [INFO] Credits remaining: 23748
2026-04-20 09:37:33 [INFO]   '越南evisa': 0 results, 0 new (total unique: 0)
2026-04-20 09:37:35 [INFO] Credits remaining: 23747
2026-04-20 09:37:35 [INFO]   '越南快速通關': 0 results, 0 new (total unique: 0)
2026-04-20 09:37:38 [INFO] Credits remaining: 23746
2026-04-20 09:37:38 [INFO]   '快速通關': 9 results, 8 new (total unique: 8)


  API回傳總筆數=17, 新增筆數=15, 重複率=11.8%

── 爬取完成 ──
  去重後貼文: 15 篇
  總輪數: 1
  API回傳總筆數(各輪加總): 17
  估算 credits 消耗: 約 5（以 Credits remaining 差值為準）
  第1輪新增: 15
  多輪累積新增: 15
  覆蓋率提升(相對第1輪): 0.0%
  各輪明細:
    R1: api=17, new=15, dup=11.8%


---

## 節點 5：載入資料 & 資料檢視

讀取爬取結果 CSV，執行基礎清洗與資料概覽。

> **語言/主題策略備註：** ScrapeCreators API 採模糊比對，無法純靠繁中關鍵字排除日/韓/泰主題貼文。本版本改以 `post_category` 標記分類（主題樂園快速通關、泰國快速通關、越南簽證/快速通關、其他），保留所有樣本供後續分析。


In [50]:
# ── 轉換層 ──
import re
from datetime import date

df = df_out.copy()

df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df['post_date'] = df['timestamp'].dt.date
df['text_length'] = df['text'].fillna('').str.len()
engagement_cols = ['like_count', 'reply_count', 'repost_count', 'quote_count', 'reshare_count']
df['total_engagement'] = df[engagement_cols].sum(axis=1)

# ── taken_at 轉台灣時間（新增欄位，不動原 Unix 秒數）──
df['taken_at_tpe'] = (
    pd.to_datetime(df['taken_at'], unit='s', utc=True)
      .dt.tz_convert('Asia/Taipei')
      .dt.strftime('%Y-%m-%d %H:%M:%S')
)


In [51]:
df

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,permalink,taken_at,timestamp,search_keyword,post_date,text_length,total_engagement,taken_at_tpe
0,3878521808000975088_71603838971,worldsimking,False,🇻🇳 飛胡志明市注意！\n\n2026 年 4 月中開始，部分外籍旅客入境新山一國際機場前，...,10,2,1,0,23,False,https://www.threads.net/@worldsimking/post/DXT...,1776575851,2026-04-19 05:17:31+00:00,越南簽證,2026-04-19,376,36,2026-04-19 13:17:31
1,3879036467197154709_69865957508,yanhe851,False,越南要簽證費 泰國免，去泰國吧,0,1,0,0,0,True,https://www.threads.net/@yanhe851/post/DXVGx1q...,1776637203,2026-04-19 22:20:03+00:00,越南簽證,2026-04-19,15,1,2026-04-20 06:20:03
2,3878450722668391454_68130912969,chenjason18,False,然後發現\n1越南工人 不能趕走\n2越南簽證 不能拒發 不然經濟會垮\n3越南茶葉不能禁止...,15,1,0,0,0,True,https://www.threads.net/@chenjason18/post/DXTB...,1776567376,2026-04-19 02:56:16+00:00,越南簽證,2026-04-19,65,16,2026-04-19 10:56:16
3,3878458489530590707_65067965811,ot_dogun,False,越南居然說台灣是中國的\n辱台太過份了\n我不去越南了\n\n這套路是不是很熟悉\n好像某國...,36,13,0,0,0,False,https://www.threads.net/@ot_dogun/post/DXTDXKA...,1776568302,2026-04-19 03:11:42+00:00,越南簽證,2026-04-19,63,49,2026-04-19 11:11:42
4,3878551417507048678_44090593522,guog.uo37,False,接下來會不會有逃逸越南仔跟留學生配合搞亂呢？,90,26,8,1,7,False,https://www.threads.net/@guog.uo37/post/DXTYfb...,1776579380,2026-04-19 06:16:20+00:00,越南簽證,2026-04-19,22,132,2026-04-19 14:16:20
5,3878413495931476621_63221150519,yougonglin,False,越南：台灣是中國的一部份。\n\n- -\n\n外交部震怒！！！然後呢？？？,1797,269,17,0,17,False,https://www.threads.net/@yougonglin/post/DXS5I...,1776562939,2026-04-19 01:42:19+00:00,越南簽證,2026-04-19,34,2100,2026-04-19 09:42:19
6,3878453359116746284_72823244840,s_loeeeee,False,？現在20萬還能整鼻子嗎？,60,60,2,0,12,False,https://www.threads.net/@s_loeeeee/post/DXTCMf...,1776567691,2026-04-19 03:01:31+00:00,越南簽證,2026-04-19,13,134,2026-04-19 11:01:31
7,3878706721678646998_65941033436,eevee520,False,想詢問有去過環球影城的各位～\n因為本人第一次去沒有去過，打算在6/16或17去，會建議有需...,4,4,1,0,0,False,https://www.threads.net/@eevee520/post/DXT7zaN...,1776597894,2026-04-19 11:24:54+00:00,快速通關,2026-04-19,70,9,2026-04-19 19:24:54
8,3878534276149574010_64228460884,tyjls4851,False,🛫 2026 日本入境超簡單攻略！Visit Japan Web 一碼搞定，快速通關不排隊！...,747,51,101,5,1895,False,https://www.threads.net/@tyjls4851/post/DXTUl_...,1776577337,2026-04-19 05:42:17+00:00,快速通關,2026-04-19,316,2799,2026-04-19 13:42:17
9,3878179682760033906_63440021940,l.guan_,False,從台灣飛荷蘭的時候排入境排超久，剛剛從英國回來發現有歐盟居留卡入境可以快速通關，隔壁all ...,175,21,0,0,27,False,https://www.threads.net/@l.guan_/post/DXSD9--DIZy,1776535066,2026-04-18 17:57:46+00:00,快速通關,2026-04-18,138,223,2026-04-19 01:57:46


In [52]:
# ── 主題分類──
THEME_PARK_TERMS = ["環球影城", "環球", "任天堂", "迪士尼", "樂天世界", "樂園"]
THAILAND_TERMS   = ["曼谷", "素萬納普", "泰國機場"]
VIETNAM_TERMS    = ["越南", "河內", "胡志明", "峴港", "Vietnam"]

def categorize_post(text: str) -> str:
    t = text or ''
    has_fast_pass = "快速通關" in t
    if has_fast_pass and any(k in t for k in THEME_PARK_TERMS):
        return "主題樂園快速通關"
    # if has_fast_pass and any(k in t for k in THAILAND_TERMS):
    #     return "泰國快速通關"
    # if any(k in t for k in VIETNAM_TERMS):
    #     return "越南簽證/快速通關"
    return "其他"

df['post_category'] = df['text'].fillna('').apply(categorize_post)


In [53]:
# ── 意圖分類──
# t1：文中含簽證/通關行動詞 → 直接意圖
# t2：其餘 → 泛旅遊/潛在意圖
T1_INTENT_PATTERN = r'簽證|電子簽|落地簽|evisa|e-visa|批文|辦簽|辦證'
T1_CONDITIONAL    = r'快速通關'
VIETNAM_CONTEXT   = r'越南|vietnam'

_text = df['text'].fillna('').str
_direct = _text.contains(T1_INTENT_PATTERN, regex=True, case=False)
_conditional = _text.contains(T1_CONDITIONAL, regex=True, case=False) & _text.contains(VIETNAM_CONTEXT, regex=True, case=False)

df['audience_type'] = np.where(_direct | _conditional, 't1', 't2')

# ── 日期範圍過濾（根據 MODE 自動使用 START_DATE / END_DATE）──
filter_start = date.fromisoformat(START_DATE)
filter_end   = date.fromisoformat(END_DATE)
x_df = df[(df['post_date'] < filter_start) | (df['post_date'] > filter_end)].copy()
df   = df[(df['post_date'] >= filter_start) & (df['post_date'] <= filter_end)].copy()
print(f"[{MODE} filter] 篩選日期: {START_DATE}" + (f" ~ {END_DATE}" if START_DATE != END_DATE else ""))
print(f"  範圍內: {len(df)} 篇，範圍外 drop: {len(x_df)} 篇")

# 標記每篇貼文命中了哪些關鍵字
for kw in KEYWORDS:
    df[f'has_{kw}'] = df['text'].fillna('').str.contains(kw, case=False)
df['matched_keywords'] = df[[f'has_{kw}' for kw in KEYWORDS]].sum(axis=1)

# # ── 繁中過濾（偵測常見簡體字，命中即視為非繁中）──
# SIMPLIFIED_CHARS = r'[国际产业发处对还时样这来应开关务办签证记单体续办证签帮验让专达进卫]'
# df['is_zh_tw'] = ~df['text'].fillna('').str.contains(SIMPLIFIED_CHARS, regex=True)
# post_non_zh_tw = df[~df['is_zh_tw']].copy()
# df = df[df['is_zh_tw']].copy()

# print(f"── 繁中過濾 ──")
# print(f"  繁中貼文: {len(df)} 篇")
# print(f"  排除簡體貼文: {len(post_non_zh_tw)} 篇")
print(f"── 意圖分類（audience_type）──")
print(f"  t1 (直接意圖): {(df['audience_type']=='t1').sum()} 篇")
print(f"  t2 (泛旅遊/潛在): {(df['audience_type']=='t2').sum()} 篇")
print(f"── 主題分類分布 ──")
# print(df['post_category'].value_counts())

[day filter] 篩選日期: 2026-04-19
  範圍內: 14 篇，範圍外 drop: 1 篇
── 意圖分類（audience_type）──
  t1 (直接意圖): 3 篇
  t2 (泛旅遊/潛在): 11 篇
── 主題分類分布 ──


In [54]:
df

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,...,total_engagement,taken_at_tpe,post_category,audience_type,has_越南簽證,has_越南電子簽,has_越南evisa,has_越南快速通關,has_快速通關,matched_keywords
0,3878521808000975088_71603838971,worldsimking,False,🇻🇳 飛胡志明市注意！\n\n2026 年 4 月中開始，部分外籍旅客入境新山一國際機場前，...,10,2,1,0,23,False,...,36,2026-04-19 13:17:31,其他,t1,False,False,False,False,False,0
1,3879036467197154709_69865957508,yanhe851,False,越南要簽證費 泰國免，去泰國吧,0,1,0,0,0,True,...,1,2026-04-20 06:20:03,其他,t1,False,False,False,False,False,0
2,3878450722668391454_68130912969,chenjason18,False,然後發現\n1越南工人 不能趕走\n2越南簽證 不能拒發 不然經濟會垮\n3越南茶葉不能禁止...,15,1,0,0,0,True,...,16,2026-04-19 10:56:16,其他,t1,True,False,False,False,False,1
3,3878458489530590707_65067965811,ot_dogun,False,越南居然說台灣是中國的\n辱台太過份了\n我不去越南了\n\n這套路是不是很熟悉\n好像某國...,36,13,0,0,0,False,...,49,2026-04-19 11:11:42,其他,t2,False,False,False,False,False,0
4,3878551417507048678_44090593522,guog.uo37,False,接下來會不會有逃逸越南仔跟留學生配合搞亂呢？,90,26,8,1,7,False,...,132,2026-04-19 14:16:20,其他,t2,False,False,False,False,False,0
5,3878413495931476621_63221150519,yougonglin,False,越南：台灣是中國的一部份。\n\n- -\n\n外交部震怒！！！然後呢？？？,1797,269,17,0,17,False,...,2100,2026-04-19 09:42:19,其他,t2,False,False,False,False,False,0
6,3878453359116746284_72823244840,s_loeeeee,False,？現在20萬還能整鼻子嗎？,60,60,2,0,12,False,...,134,2026-04-19 11:01:31,其他,t2,False,False,False,False,False,0
7,3878706721678646998_65941033436,eevee520,False,想詢問有去過環球影城的各位～\n因為本人第一次去沒有去過，打算在6/16或17去，會建議有需...,4,4,1,0,0,False,...,9,2026-04-19 19:24:54,主題樂園快速通關,t2,False,False,False,False,True,1
8,3878534276149574010_64228460884,tyjls4851,False,🛫 2026 日本入境超簡單攻略！Visit Japan Web 一碼搞定，快速通關不排隊！...,747,51,101,5,1895,False,...,2799,2026-04-19 13:42:17,其他,t2,False,False,False,False,True,1
10,3878524114424734693_68711406883,dsproject2024,False,售 日本環球影城1.5日門票+快速通關\n\n早早規劃的家族旅遊，無奈計畫還是趕不上變化。 ...,0,1,0,0,0,False,...,1,2026-04-19 13:22:05,主題樂園快速通關,t2,False,False,False,False,True,1


In [55]:
x_df

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,permalink,taken_at,timestamp,search_keyword,post_date,text_length,total_engagement,taken_at_tpe,post_category,audience_type
9,3878179682760033906_63440021940,l.guan_,False,從台灣飛荷蘭的時候排入境排超久，剛剛從英國回來發現有歐盟居留卡入境可以快速通關，隔壁all ...,175,21,0,0,27,False,https://www.threads.net/@l.guan_/post/DXSD9--DIZy,1776535066,2026-04-18 17:57:46+00:00,快速通關,2026-04-18,138,223,2026-04-19 01:57:46,其他,t2


In [56]:
# ── 商案文標記──
COMMERCIAL_TERMS = [
    "LINE ID", "LINE", "加入LINE", "加我", "歡迎諮詢", "招募", "互追", "漲粉",
    "立即預訂", "車隊", "限時", "超時費", "客製",
    " 點擊主頁連結",  "點擊主頁",  "點擊連結",
    "經營", "急單", "加急", "特急", "服務品質", "服務客人", "我的客人", "幫客人", "貴賓", "提前預約", "團體優惠",
    "免費幫忙", "免費協助", "免費諮詢", "為您", "領取折扣", "領取優惠券", "免費領取", "保證過件", "本公司", "我司",
    "留言", "傳給你", "歡迎詢問", "歡迎私訊", "私訊了解", "私訊", "洽詢我們", "聯繫我們", "聯繫我", "洽詢", "請洽", "可以追蹤", "找我們", "交給我們"
]

commercial_pattern = '|'.join(COMMERCIAL_TERMS)

# 商家聯繫方式 regex（高訊噪比，一般用戶幾乎不會出現）
COMMERCIAL_REGEXES = [
    r"lin\.ee/[A-Za-z0-9]+",        # LINE 官方帳號短網址
    r"\d{4,}\s?起[/／]人",            # 旅行社團費「19900起/人」
    r"官方LINE[:：@]",                # 「官方LINE:@xxx」
]

def is_commercial(text: str) -> bool:
    t = text or ''
    if re.search(commercial_pattern, t, flags=re.IGNORECASE):
        return True
    return any(re.search(p, t, flags=re.IGNORECASE) for p in COMMERCIAL_REGEXES)

df['is_commercial'] = df['text'].fillna('').apply(is_commercial)

# 商案文 → 覆寫 post_category 為「商案文」
df.loc[df['is_commercial'], 'post_category'] = '商案文'

# 不再排除，全部保留進 df_clean
df_clean = df.copy()

print(f"── 商案文標記 ──")
print(f"  商案文: {df['is_commercial'].sum()} 篇（已標記 post_category='商案文'）")
print(f"  一般文: {(~df['is_commercial']).sum()} 篇")
print(f"  df_clean 總計: {len(df_clean)} 篇（全部保留）")
print(f"\n── post_category 分布 ──")
print(df_clean['post_category'].value_counts().to_string())

── 商案文標記 ──
  商案文: 4 篇（已標記 post_category='商案文'）
  一般文: 10 篇
  df_clean 總計: 14 篇（全部保留）

── post_category 分布 ──
post_category
其他          8
商案文         4
主題樂園快速通關    2


In [57]:
df_clean

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,...,taken_at_tpe,post_category,audience_type,has_越南簽證,has_越南電子簽,has_越南evisa,has_越南快速通關,has_快速通關,matched_keywords,is_commercial
0,3878521808000975088_71603838971,worldsimking,False,🇻🇳 飛胡志明市注意！\n\n2026 年 4 月中開始，部分外籍旅客入境新山一國際機場前，...,10,2,1,0,23,False,...,2026-04-19 13:17:31,商案文,t1,False,False,False,False,False,0,True
1,3879036467197154709_69865957508,yanhe851,False,越南要簽證費 泰國免，去泰國吧,0,1,0,0,0,True,...,2026-04-20 06:20:03,其他,t1,False,False,False,False,False,0,False
2,3878450722668391454_68130912969,chenjason18,False,然後發現\n1越南工人 不能趕走\n2越南簽證 不能拒發 不然經濟會垮\n3越南茶葉不能禁止...,15,1,0,0,0,True,...,2026-04-19 10:56:16,其他,t1,True,False,False,False,False,1,False
3,3878458489530590707_65067965811,ot_dogun,False,越南居然說台灣是中國的\n辱台太過份了\n我不去越南了\n\n這套路是不是很熟悉\n好像某國...,36,13,0,0,0,False,...,2026-04-19 11:11:42,其他,t2,False,False,False,False,False,0,False
4,3878551417507048678_44090593522,guog.uo37,False,接下來會不會有逃逸越南仔跟留學生配合搞亂呢？,90,26,8,1,7,False,...,2026-04-19 14:16:20,其他,t2,False,False,False,False,False,0,False
5,3878413495931476621_63221150519,yougonglin,False,越南：台灣是中國的一部份。\n\n- -\n\n外交部震怒！！！然後呢？？？,1797,269,17,0,17,False,...,2026-04-19 09:42:19,其他,t2,False,False,False,False,False,0,False
6,3878453359116746284_72823244840,s_loeeeee,False,？現在20萬還能整鼻子嗎？,60,60,2,0,12,False,...,2026-04-19 11:01:31,其他,t2,False,False,False,False,False,0,False
7,3878706721678646998_65941033436,eevee520,False,想詢問有去過環球影城的各位～\n因為本人第一次去沒有去過，打算在6/16或17去，會建議有需...,4,4,1,0,0,False,...,2026-04-19 19:24:54,主題樂園快速通關,t2,False,False,False,False,True,1,False
8,3878534276149574010_64228460884,tyjls4851,False,🛫 2026 日本入境超簡單攻略！Visit Japan Web 一碼搞定，快速通關不排隊！...,747,51,101,5,1895,False,...,2026-04-19 13:42:17,其他,t2,False,False,False,False,True,1,False
10,3878524114424734693_68711406883,dsproject2024,False,售 日本環球影城1.5日門票+快速通關\n\n早早規劃的家族旅遊，無奈計畫還是趕不上變化。 ...,0,1,0,0,0,False,...,2026-04-19 13:22:05,商案文,t2,False,False,False,False,True,1,True


In [58]:
# ── 資料觀察 ──
print("=== 資料概覽 ===")
print(f"  貼文數: {len(df_clean)}")
print(f"  不重複帳號: {df_clean['username'].nunique()}")
print(f"  時間範圍: {df_clean['timestamp'].min():%Y-%m-%d} ~ {df_clean['timestamp'].max():%Y-%m-%d}")
print(f"  is_reply 分布: {df_clean['is_reply'].value_counts().to_dict()}")

print(f"\n=== search_keyword 來源分布 ===")
print(df_clean['search_keyword'].value_counts().to_string())

print(f"\n=== 互動指標描述統計 ===")
print(df_clean[engagement_cols + ['total_engagement']].describe().round(1).to_string())

=== 資料概覽 ===
  貼文數: 14
  不重複帳號: 14
  時間範圍: 2026-04-19 ~ 2026-04-19
  is_reply 分布: {False: 12, True: 2}

=== search_keyword 來源分布 ===
search_keyword
越南簽證    7
快速通關    7

=== 互動指標描述統計 ===
       like_count  reply_count  repost_count  quote_count  reshare_count  total_engagement
count        14.0         14.0          14.0         14.0           14.0              14.0
mean        202.1         34.2           9.3          0.4          143.1             389.2
std         498.5         71.0          26.8          1.3          504.4             885.0
min           0.0          0.0           0.0          0.0            0.0               1.0
25%           4.5          1.0           0.0          0.0            0.0               8.2
50%          17.0          3.0           0.0          0.0            3.5              30.0
75%          56.0         43.2           1.8          0.0           15.8             133.5
max        1797.0        269.0         101.0          5.0         1895.0            279

In [59]:
# # ── 輸出清洗後 CSV/JSON ──
# csv_path = DATA_DIR / f"threads_vietnam_visa_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
# json_path = csv_path.with_suffix('.json')

# df_clean.to_csv(csv_path, index=False, encoding='utf-8')
# save_json(df_clean.to_dict(orient='records'), str(json_path))

# print(f"[OK] {csv_path.name}")
# print(f"[OK] {json_path.name}")


In [60]:
# # 測試上傳gshhet
# test_csv_path = DATA_DIR / "threads_vietnam_visa_backfill_20260401_20260415.csv"
# test_csv = pd.read_csv(test_csv_path, encoding="utf-8")

# # 後面的 Google Sheets 上傳 cell 讀取 df_clean，測試時用回補 CSV 覆蓋 df_clean。
# df_clean = test_csv.copy()

# print(f"[OK] 載入測試 CSV: {test_csv_path.resolve()}")
# print(f"[OK] df_clean 已切換為測試資料，共 {len(df_clean)} 筆")

In [61]:
# ── 上傳到 Google Sheets ──
import gspread
from google.oauth2.service_account import Credentials

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
CREDS_PATH = Path("ads-de-01-757fc521ef01.json")
SHEET_ID = "1jgZI7nsdHxnTVOReytzCkINYGR-f8SEW3kJzZzOa7ts"
WORKSHEET_GID = 855855256

creds = Credentials.from_service_account_file(CREDS_PATH, scopes=SCOPES)
gc = gspread.authorize(creds)

sh = gc.open_by_key(SHEET_ID)
ws = sh.get_worksheet_by_id(WORKSHEET_GID)
print(f"[OK] 已連線 worksheet: {ws.title}")


[OK] 已連線 worksheet: vn_visa_service


In [62]:
# 篩選欄位
ouput_df = df_clean[[
    'username', 'taken_at_tpe', 'post_date', 'text',
    'like_count', 'reply_count', 'repost_count', 'quote_count', 'reshare_count',
    'permalink', 'search_keyword', 'audience_type', 'post_category', 
]].copy()
ouput_df['scrape_date'] = datetime.now().strftime('%Y-%m-%d')

# ── 安全寫入：明確定位最後一行，不依賴 append_rows 的表格偵測 ──
existing = ws.get_all_values()
next_row = len(existing) + 1

# 空表：先寫標題到 A1
if not existing or not any(str(cell).strip() for cell in existing[0]):
    ws.update(range_name='A1', values=[ouput_df.columns.tolist()], value_input_option="RAW")
    next_row = 2
    print(f"[init] 已寫入標題列")

# 明確寫入 A{next_row}，繞過 Google Sheets 表格邊界偵測
data_rows = ouput_df.astype(str).values.tolist()

if data_rows:
    required_rows = next_row + len(data_rows) - 1
    required_cols = len(ouput_df.columns)
    target_rows = max(ws.row_count, required_rows)
    target_cols = max(ws.col_count, required_cols)

    if target_rows > ws.row_count or target_cols > ws.col_count:
        ws.resize(rows=target_rows, cols=target_cols)
        print(f"[resize] Sheet grid 已調整為 rows={target_rows}, cols={target_cols}")

    ws.update(range_name=f'A{next_row}', values=data_rows, value_input_option="RAW")

    print(f"[OK] 已寫入 {len(data_rows)} 筆到 A{next_row}~A{required_rows}")
    print(f"  Sheet 目前共 {required_rows} 行（含標題）")
    print(f"  https://docs.google.com/spreadsheets/d/{SHEET_ID}")
else:
    print("[skip] ouput_df 無資料，略過 Google Sheets 寫入")

[OK] 已寫入 14 筆到 A379~A392
  Sheet 目前共 392 行（含標題）
  https://docs.google.com/spreadsheets/d/1jgZI7nsdHxnTVOReytzCkINYGR-f8SEW3kJzZzOa7ts


In [66]:
backup_csv_path = DATA_DIR / f"threads_vn_visa_post_{(datetime.now() - timedelta(days=1)).strftime('%Y%m%d')}.csv"
ouput_df.to_csv(backup_csv_path, index=False, encoding='utf-8-sig')
print(f"[OK] 已備份 CSV: {backup_csv_path}")

[OK] 已備份 CSV: data\threads_vn_visa_post_20260419.csv
